# Activity 4: Mentor Matching & Intervetion Recommendations

## Objective

Translate insights from students dataset into real mentoring actions. This notebook demonstrates the student-mentor matching based on the mentorship needs of each student and intervetion type provided by mentor on heuristc basis.  

In [1]:
import os 
from pathlib import Path

In [2]:
root_path = Path.cwd().parent
os.chdir(root_path)

In [3]:
import pandas as pd
from src.models.mentor_matching import match_mentor

## Loading data

In [4]:
students = pd.read_csv("data/processed/students_clustered.csv")
mentors = pd.read_csv("data/raw/mentors.csv")

In [5]:
mentors.head()

,mentor_id,name,expertise,max_capacity,current_load,availability_hours,priority_levels
0,M001,Dr. Sharma,academic,25,10,15,Blue;Yellow
1,M002,Ms. Kapoor,career,20,8,12,Yellow;Red
2,M003,Mr. Iyer,productivity,18,5,10,Blue;Yellow;Red
3,M004,Dr. Sen,wellness,15,7,8,Red
4,M005,Ms. Rao,academic,20,4,14,Blue;Yellow


In [6]:
students.head(5)

,student_id,age,program,semester,gpa,attendance,assignment_completion,engagement_score,stress_level,career_clarity,...,productivity_score,distractions,skill_readiness,APS,WWS,PTMS,CRS,SRI,risk_category,cluster
0,S001,20,B.Tech,7,8.5,44.2,93.3,35.9,6,5,...,7,2,4,79.33,44.8,74.0,45.0,61.05,Yellow,0
1,S002,20,B.Tech,2,8.7,35.5,90.8,34.2,5,7,...,7,3,3,77.84,56.0,70.0,50.0,63.85,Yellow,0
2,S003,19,MBA,4,8.4,31.8,83.2,43.8,5,5,...,7,2,4,73.32,50.0,74.0,45.0,60.55,Yellow,0
3,S004,18,BCA,3,8.5,25.4,82.0,43.4,6,5,...,6,5,3,72.18,47.8,56.0,40.0,54.80,Yellow,0
4,S005,19,BCA,2,9.4,41.3,93.3,35.1,6,6,...,7,4,3,83.25,47.8,66.0,45.0,61.38,Yellow,0


## Recommendation

In [7]:
recommendations = match_mentor(students, mentors)

In [8]:
recommendations.head()

,student_id,cluster,risk_category,mentor_id,mentor_name,mentor_expertise,intervention,assignment_status,alert
0,S001,0,Yellow,M002,Ms. Kapoor,career,Career Direction,Assigend,False
1,S002,0,Yellow,M002,Ms. Kapoor,career,Career Direction,Assigend,False
2,S003,0,Yellow,M002,Ms. Kapoor,career,Career Direction,Assigend,False
3,S004,0,Yellow,M002,Ms. Kapoor,career,Career Direction,Assigend,False
4,S005,0,Yellow,M002,Ms. Kapoor,career,Career Direction,Assigend,False


In [9]:
recommendations.shape

(200, 9)

In [10]:
recommendations['mentor_name'].value_counts()

mentor_name
Pending Assignment    136
Ms. Rao                16
Dr. Sharma             15
Mr. Iyer               13
Ms. Kapoor             12
Dr. Sen                 8
Name: count, dtype: int64

## Persisting the data

In [11]:
recommendations.to_csv("data/processed/recommendations.csv", index=False)
print(f"Recommendations saved!")

Recommendations saved!


$\quad$

---

$\quad$

# Hybrid Mentor Matching using Cosine Similarity

## Motivation

The initial mentor matching system used rule-based heuristics considering
cluster, risk level, mentor expertise, and capacity constraints.

While effective, this approach may assign any available mentor within
constraints without evaluating how well a mentor’s strengths align with
a student’s specific needs.

To improve personalization, a similarity-based ranking mechanism is added.
Cosine similarity is used to compare student need vectors with mentor
capability vectors.

The final approach is hybrid:

**Rule-based filtering → Cosine similarity ranking → Best mentor selection**

This preserves operational constraints while improving match quality.

## Vector Representation

Both students and mentors are represented in a shared capability space:

[Academic Support, Wellness Support, Productivity Coaching, Career Guidance]

### Student Need Vectors (derived from cluster)

Cluster 0 — Academic Performers:
High academic optimization and career planning needs.

Cluster 1 — Directionless Students:
High need for productivity structure and career guidance.

Cluster 2 — Burnout Students:
High need for wellness intervention.

### Mentor Capability Vectors

Mentor expertise is mapped to strengths across the same dimensions.

Cosine similarity measures alignment between student needs and mentor capabilities.

In [12]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
from src.models.mentor_cosine_match import select_mentor_cosine, match_students_cosine

In [22]:
# priority logic, urgent student cases must be dealt first
RISK_PRIORITY = {
    "Red": 0,
    "Yellow": 1,
    "Blue": 2,
    "Green": 3
}

CLUSTER_PRIORITY = {
    2: 0,  # Burnout
    1: 1,  # Directionless
    0: 2,  # Academic Performers
}

In [29]:
students_clustered = pd.read_csv("data/processed/students_clustered.csv")
mentors_for_cosine = pd.read_csv("data/raw/mentors.csv")

# sort the students as per PRIORITY Settings
students_prioritized = students_clustered.copy()

students_prioritized["risk_priority"] = students_prioritized['risk_category'].map(RISK_PRIORITY)

students_prioritized['cluster_priority'] = students_prioritized['cluster'].map(CLUSTER_PRIORITY)

students_prioritized.sort_values(by=["risk_priority"])

students_prioritized.head()

,student_id,age,program,semester,gpa,attendance,assignment_completion,engagement_score,stress_level,career_clarity,...,skill_readiness,APS,WWS,PTMS,CRS,SRI,risk_category,cluster,risk_priority,cluster_priority
0,S001,20,B.Tech,7,8.5,44.2,93.3,35.9,6,5,...,4,79.33,44.8,74.0,45.0,61.05,Yellow,0,1,2
1,S002,20,B.Tech,2,8.7,35.5,90.8,34.2,5,7,...,3,77.84,56.0,70.0,50.0,63.85,Yellow,0,1,2
2,S003,19,MBA,4,8.4,31.8,83.2,43.8,5,5,...,4,73.32,50.0,74.0,45.0,60.55,Yellow,0,1,2
3,S004,18,BCA,3,8.5,25.4,82.0,43.4,6,5,...,3,72.18,47.8,56.0,40.0,54.80,Yellow,0,1,2
4,S005,19,BCA,2,9.4,41.3,93.3,35.1,6,6,...,3,83.25,47.8,66.0,45.0,61.38,Yellow,0,1,2


In [33]:
cosine_recommendations = match_students_cosine(students_prioritized, mentors_for_cosine)

cosine_recommendations.head()

,student_id,cluster,risk_category,mentor_id,mentor_name,assignment_status
0,S001,0,Yellow,M001,Dr. Sharma,Assigned
1,S002,0,Yellow,M001,Dr. Sharma,Assigned
2,S003,0,Yellow,M001,Dr. Sharma,Assigned
3,S004,0,Yellow,M001,Dr. Sharma,Assigned
4,S005,0,Yellow,M001,Dr. Sharma,Assigned


In [35]:
cosine_recommendations["mentor_name"].value_counts()

mentor_name
Pending Assignment    136
Ms. Rao                16
Dr. Sharma             15
Mr. Iyer               13
Ms. Kapoor             12
Dr. Sen                 8
Name: count, dtype: int64

In [32]:
# persisting the cosine recommendations

cosine_recommendations.to_csv("data/processed/cosine_recommendation.csv", index=False)
print("Cosine Recommendation saved!")

Cosine Recommendation saved!
